# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes
## 1. Ranked actions + reason codes

The action queue prioritizes content items for human review using observable GSC signals.

The recommendations are decision-support signals, not automatic content changes.

Each item receives a reason code that explains why it was prioritized:

* `HIGH_IMPRESSIONS_LOW_CTR`: high exposure with relatively low CTR.
* `GOOD_POSITION_LOW_CTR`: reasonable search position with relatively low CTR.
* `LOW_VISIBILITY`: lower search visibility that may justify further review.
* `GENERAL_REVIEW`: does not match a stronger reason code but remains a lower-priority review candidate.

The queue is ranked using observable performance signals and is intended to help a reviewer decide where to investigate first.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 1. Ranked actions + reason codes

# Section 1 — Ranked actions + reason codes

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

# --------------------------------------------------
# 1. Hugging Face authentication
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face token as HF_TOKEN."
    )

# --------------------------------------------------
# 2. DuckDB connection
# --------------------------------------------------

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

# --------------------------------------------------
# 3. Dataset path
# --------------------------------------------------

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-05/data_0.parquet"
)

# --------------------------------------------------
# 4. Load data
# --------------------------------------------------

queue_query = f"""
SELECT
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    gsc_clicks * 100.0
        / NULLIF(gsc_impressions, 0) AS ctr

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
AND gsc_impressions > 0
AND gsc_avg_position > 0
"""

queue_df = con.execute(queue_query).fetchdf()

print("Rows loaded:", len(queue_df))

# --------------------------------------------------
# 5. Clean data
# --------------------------------------------------

queue_df = queue_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr"
    ]
)

# --------------------------------------------------
# 6. Reason codes
# --------------------------------------------------

def assign_reason(row):

    if (
        row["gsc_impressions"] >= 500
        and row["gsc_avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        return "HIGH_IMPRESSIONS_LOW_CTR"

    elif (
        row["gsc_avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        return "GOOD_POSITION_LOW_CTR"

    elif row["gsc_avg_position"] > 20:
        return "LOW_VISIBILITY"

    else:
        return "GENERAL_REVIEW"


queue_df["reason_code"] = queue_df.apply(
    assign_reason,
    axis=1
)

# --------------------------------------------------
# 7. Recommended actions
# --------------------------------------------------

action_map = {
    "HIGH_IMPRESSIONS_LOW_CTR":
        "Review title and search snippet",

    "GOOD_POSITION_LOW_CTR":
        "Review search intent, title, and snippet",

    "LOW_VISIBILITY":
        "Review content relevance and search visibility",

    "GENERAL_REVIEW":
        "Perform general content review"
}

queue_df["recommended_action"] = (
    queue_df["reason_code"].map(action_map)
)

# --------------------------------------------------
# 8. Priority
# --------------------------------------------------

priority_map = {
    "HIGH_IMPRESSIONS_LOW_CTR": "High",
    "GOOD_POSITION_LOW_CTR": "Medium",
    "LOW_VISIBILITY": "Medium",
    "GENERAL_REVIEW": "Low"
}

queue_df["priority"] = queue_df["reason_code"].map(
    priority_map
)

priority_score = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}

queue_df["priority_score"] = queue_df["priority"].map(
    priority_score
)

# --------------------------------------------------
# 9. Rank the queue
# --------------------------------------------------

queue_df = queue_df.sort_values(
    by=[
        "priority_score",
        "gsc_impressions"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

queue_df["rank"] = np.arange(
    1,
    len(queue_df) + 1
)

# --------------------------------------------------
# 10. Final queue
# --------------------------------------------------

ranked_queue = queue_df[
    [
        "rank",
        "content_hash_id",
        "report_date",
        "priority",
        "reason_code",
        "recommended_action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position"
    ]
].copy()

display(ranked_queue.head(20))

print("\nQueue size:", len(ranked_queue))

print("\nReason codes:")
print(
    ranked_queue["reason_code"].value_counts()
)

print("\nSection 1 check passed.")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 4237020


,rank,content_hash_id,report_date,priority,reason_code,recommended_action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,content_cedadfe5ae4845ac,2026-05-26,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,47607,57,0.119730,5.632722
1,2,content_21309e9a83c83653,2026-05-25,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,46325,58,0.125202,5.071689
2,3,content_65c75874a23fca87,2026-05-17,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,32168,2,0.006217,8.426604
3,4,content_2f567cb6ad16f6f0,2026-05-25,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,28431,3,0.010552,9.095741
4,5,content_545bb6cc7081ded3,2026-05-26,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,26663,84,0.315043,2.153734
5,6,content_86c96002dd5c69aa,2026-05-05,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,25922,62,0.239179,10.144433
6,7,content_21309e9a83c83653,2026-05-24,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,25806,25,0.096877,5.178602
7,8,content_21309e9a83c83653,2026-05-26,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,24429,38,0.155553,5.006877
8,9,content_65c75874a23fca87,2026-05-18,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,24391,2,0.008200,8.963142
9,10,content_545bb6cc7081ded3,2026-05-27,High,HIGH_IMPRESSIONS_LOW_CTR,Review title and search snippet,22798,85,0.372840,2.148083



Queue size: 4237020

Reason codes:
reason_code
GOOD_POSITION_LOW_CTR       2296420
LOW_VISIBILITY              1566164
GENERAL_REVIEW               306774
HIGH_IMPRESSIONS_LOW_CTR      67662
Name: count, dtype: int64

Section 1 check passed.


## 2. Intended use and limits

## 2. Intended use and limits

### Intended use

The playbook is intended to help content teams prioritize content items for human review.

The ranked queue uses observed GSC performance signals such as impressions, clicks, CTR, and average position to identify items that may deserve further investigation.

The output is a decision-support tool. It helps a reviewer decide where to look first rather than making the final content decision.

### Limits

The recommendations are directional and do not guarantee that a specific content change will improve CTR or business performance.

The current target and feature design have a close dependency, as identified in the previous validation audit. Therefore, the measured results should be interpreted cautiously.

The playbook is not intended to automatically publish, modify, delete, or redirect content.

Human judgment and additional context are required before taking action.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Ranked actions + reason codes

# Section 2 — Intended use and limits check

intended_use = {
    "purpose": "Prioritize content items for human review",
    "role": "Decision-support",
    "automation": "No automatic content changes",
    "claim_type": "Observed and directional signals"
}

limits = [
    "Recommendations do not guarantee CTR improvement",
    "Current target and feature dependency limits interpretation",
    "Human review is required before action",
    "The playbook is not a production automation system"
]

print("Intended use:")
for key, value in intended_use.items():
    print(f"- {key}: {value}")

print("\nLimits:")
for item in limits:
    print(f"- {item}")

assert intended_use["role"] == "Decision-support"
assert intended_use["automation"] == "No automatic content changes"
assert len(limits) >= 4

print("\nSection 2 check passed.")

Intended use:
- purpose: Prioritize content items for human review
- role: Decision-support
- automation: No automatic content changes
- claim_type: Observed and directional signals

Limits:
- Recommendations do not guarantee CTR improvement
- Current target and feature dependency limits interpretation
- Human review is required before action
- The playbook is not a production automation system

Section 2 check passed.


## 3. Human review + the no-go list

## 3. Human review + the no-go list

### Human review rules

Before acting on a recommendation, a human reviewer should:

1. Confirm the content item's search context.
2. Check the current search intent.
3. Review the existing title and search snippet.
4. Check content relevance and quality.
5. Consider business and editorial context.
6. Confirm that the recommendation is supported by the observed data.
7. Record the final decision.

High-priority recommendations should receive human review before any content change.

### No-go list

The following actions should not be automated:

* Automatically publishing content changes.
* Automatically rewriting titles or snippets.
* Automatically deleting content.
* Automatically redirecting URLs.
* Automatically changing URLs.
* Automatically making business or commercial decisions.
* Automatically claiming that a content change will improve CTR.
* Automatically overriding editorial or subject-matter decisions.

The model output is a prioritization signal, not an autonomous decision.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Human review and no-go list

human_review_rules = [
    "Confirm search context",
    "Check search intent",
    "Review title and search snippet",
    "Check content relevance and quality",
    "Consider business and editorial context",
    "Confirm evidence from observed data",
    "Record the final human decision"
]

no_go_actions = [
    "Automatically publish content changes",
    "Automatically rewrite titles or snippets",
    "Automatically delete content",
    "Automatically redirect URLs",
    "Automatically change URLs",
    "Automatically make business decisions",
    "Automatically claim CTR improvement",
    "Automatically override editorial decisions"
]

print("Human review rules:")
for rule in human_review_rules:
    print("✓", rule)

print("\nNo-go automation list:")
for action in no_go_actions:
    print("✗", action)

assert len(human_review_rules) >= 5
assert len(no_go_actions) >= 5

print("\nSection 3 check passed.")

Human review rules:
✓ Confirm search context
✓ Check search intent
✓ Review title and search snippet
✓ Check content relevance and quality
✓ Consider business and editorial context
✓ Confirm evidence from observed data
✓ Record the final human decision

No-go automation list:
✗ Automatically publish content changes
✗ Automatically rewrite titles or snippets
✗ Automatically delete content
✗ Automatically redirect URLs
✗ Automatically change URLs
✗ Automatically make business decisions
✗ Automatically claim CTR improvement
✗ Automatically override editorial decisions

Section 3 check passed.


## 4. Monitoring / retrain triggers

## 4. Monitoring / retrain triggers

The action playbook should be monitored because search behavior and content performance can change over time.

### Monitoring triggers

The following signals should be reviewed regularly:

* Distribution of priority levels.
* Distribution of reason codes.
* Changes in impressions, CTR, and average position.
* Changes in the input feature distributions.
* Changes in model performance when labeled outcomes become available.
* New or persistent error patterns.

### Retrain or review triggers

A model review or retraining cycle should be considered when:

* Measured validation performance decreases materially.
* Input feature distributions change substantially.
* The recommendation distribution changes unexpectedly.
* New labeled data becomes available.
* A persistent new failure pattern appears.

These are monitoring and review triggers, not automatic retraining commands.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Monitoring / retrain triggers

monitoring_triggers = {
    "priority_distribution": True,
    "reason_code_distribution": True,
    "performance_metrics": True,
    "feature_distribution": True,
    "error_patterns": True
}

retrain_triggers = {
    "material_performance_drop": True,
    "feature_distribution_shift": True,
    "unexpected_recommendation_shift": True,
    "new_labeled_data": True,
    "persistent_new_error_pattern": True
}

print("Monitoring triggers:")
for trigger, enabled in monitoring_triggers.items():
    print(f"✓ {trigger}: {enabled}")

print("\nRetrain/review triggers:")
for trigger, enabled in retrain_triggers.items():
    print(f"✓ {trigger}: {enabled}")

assert all(monitoring_triggers.values())
assert all(retrain_triggers.values())

print("\nSection 4 check passed.")

Monitoring triggers:
✓ priority_distribution: True
✓ reason_code_distribution: True
✓ performance_metrics: True
✓ feature_distribution: True
✓ error_patterns: True

Retrain/review triggers:
✓ material_performance_drop: True
✓ feature_distribution_shift: True
✓ unexpected_recommendation_shift: True
✓ new_labeled_data: True
✓ persistent_new_error_pattern: True

Section 4 check passed.


## 5. Exports for the paper

## 5. Exports for the paper

The ranked action queue is exported so that the next week's research paper can reuse the same generated output.

The queue CSV is written to `work/outputs/`. It is a generated data artifact and should remain outside Git according to the repository rules.

A small metrics JSON is also generated to provide traceable summary numbers for the paper.

The notebook regenerates these files whenever it is executed.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5 — Export queue and metrics

import os
import json

OUTPUT_DIR = "work/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# Export ranked queue
# --------------------------------------------------

QUEUE_PATH = os.path.join(
    OUTPUT_DIR,
    "w07_ranked_action_queue.csv"
)

ranked_queue.to_csv(
    QUEUE_PATH,
    index=False
)

# --------------------------------------------------
# Create metrics
# --------------------------------------------------

metrics = {
    "queue_rows": int(len(ranked_queue)),
    "high_priority_rows": int(
        (ranked_queue["priority"] == "High").sum()
    ),
    "medium_priority_rows": int(
        (ranked_queue["priority"] == "Medium").sum()
    ),
    "low_priority_rows": int(
        (ranked_queue["priority"] == "Low").sum()
    ),
    "reason_code_counts": {
        str(key): int(value)
        for key, value in
        ranked_queue["reason_code"].value_counts().items()
    }
}

METRICS_PATH = os.path.join(
    OUTPUT_DIR,
    "w07_action_playbook_metrics.json"
)

with open(METRICS_PATH, "w") as file:
    json.dump(
        metrics,
        file,
        indent=2
    )

print("Queue exported:")
print(QUEUE_PATH)

print("\nMetrics exported:")
print(METRICS_PATH)

print("\nMetrics:")
print(json.dumps(metrics, indent=2))

print("\nSection 5 check passed.")

Queue exported:
work/outputs/w07_ranked_action_queue.csv

Metrics exported:
work/outputs/w07_action_playbook_metrics.json

Metrics:
{
  "queue_rows": 4237020,
  "high_priority_rows": 67662,
  "medium_priority_rows": 3862584,
  "low_priority_rows": 306774,
  "reason_code_counts": {
    "GOOD_POSITION_LOW_CTR": 2296420,
    "LOW_VISIBILITY": 1566164,
    "GENERAL_REVIEW": 306774,
    "HIGH_IMPRESSIONS_LOW_CTR": 67662
  }
}

Section 5 check passed.


## 6. 5-Minute Demo Outline

### 0:00–0:45 — The question

Can observable search-performance signals help a content team prioritize which content items deserve human review first?

The goal is decision support, not automatic content optimization.

### 0:45–1:45 — The method

I used observable Google Search Console performance signals from the available FlyRank dataset, including impressions, clicks, CTR, and average position.

The workflow combined:
- A rule-based baseline
- Signal and leakage audits
- A grouped validation design
- Logistic Regression modeling
- A ranked action playbook

The evaluation used a client-grouped holdout so that clients in the test set were not used during training.

### 1:45–2:45 — One chart

Show the model-versus-baseline performance comparison from the capstone analysis.

The main metric is F1 because the opportunity class is relatively uncommon and accuracy alone can be misleading.

### 2:45–3:45 — One honest result

The Week-4 rule baseline achieved an F1 score of 1.0000 on the defined holdout, while Logistic Regression achieved an F1 score of 0.6584.

The model also produced high recall (0.9908) but lower precision (0.493).

These results should be interpreted cautiously because the target is derived from the same GSC signals used as model features.

The evidence is observational and directional rather than causal.

### 3:45–5:00 — Recommendation

Use the ranked action queue as a human-review prioritization tool.

High-priority items with high impressions, good position, and low CTR should be reviewed first for title, snippet, and search-intent alignment.

The system should not automatically rewrite, publish, delete, redirect, or make business decisions.

The recommended next step is to validate promising recommendations with editorial review and, where possible, future labeled outcomes.

## 7. Shareable Cuts

### Short social post

I built a machine-learning workflow for content-performance prioritization using observable search-performance data.

The project combines leakage checks, signal audits, a client-grouped validation split, Logistic Regression, and a ranked action playbook.

One important lesson was that strong-looking model results still need careful interpretation when the target is closely related to the same signals used as features.

The final output is decision support: it helps a content team decide what to review first rather than automatically changing content.

### Employer-facing summary

I built a content-performance decision-support workflow using search-performance data with impressions, clicks, CTR, and average position.

It compared a rule-based baseline with Logistic Regression using a client-grouped holdout and produced a ranked action queue with human-review recommendations.

The analysis showed strong recall but also highlighted important limitations around target-feature dependency, so the results are presented as observed and directional rather than causal claims.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.